# Marcha Blanca: Protocolo de Evaluación y Shadow Testing (v3.0 / v2.5)
**TuBoleta - Data Science & Machine Learning**

### Ficha Técnica de la Evaluación:
- **Propósito:** Protocolo oficial de Marcha Blanca (*Shadow Testing*) para evaluar el modelo de clusterización jerárquica de localidades (Nivel 1 Macro-Arquetipos v2.5 + Nivel 2 Micro-Clusters Canónicos v3.0) sobre nuevos eventos curados de Secutix sin impacto en producción.
- **Path de Origen (Blob de Test):** `GOLD/SECUTIX/Training Data/Clustering de Localidades test/`
- **Versión del Modelo Evaluado:** `v3.0` (jerárquico, 19 micro-clusters) / `v2.5` (arquetipos de demanda de producción)
- **Fecha de Evaluación:** `2026-10-07`
- **Artefactos Producidos:** `reports/marcha_blanca_20261007.json`, `data/processed/marcha_blanca_predicciones.csv`

### Fases del Protocolo:
1. **Carga Estándar:** Descarga de particiones de prueba desde Azure Blob Storage vía `src.azure_utils.cargar_parquet_desde_azure`.
2. **4 Checks de Validación de Entrada:** Duplicados en crudo, conteo post-filtro por regla, coherencia de aforos y volumetría.
3. **Ruta Estándar de Enriquecimiento:** Invocación de `src.feature_engineering.preparar_dataset_enriquecido` (filtro de consistencia + venue + métricas relativas + NLP).
4. **Inferencia Productiva:** Aplicación de los envoltorios vigentes `predecir_arquetipos_demanda` y `predecir_microclusters` (cero re-entrenamiento).
5. **Auditoría Cuantitativa:** Comparación de distribución observada vs. esperada, tasa de frontera, score de confianza y persistencia de artefactos.

In [ ]:
# 1. Configuración de Entorno e Importación de Módulos del Repositorio
import os
import sys
import json
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Ajuste de paths para soportar ejecución en Databricks Repos y local
for p in ['.', '..']:
    if p not in sys.path:
        sys.path.insert(0, p)

env_path = next((p for p in ['../.env', '.env'] if os.path.exists(p)), None)
if env_path:
    load_dotenv(env_path)
    print(f"Credenciales cargadas desde: {env_path}")

from src.azure_utils import cargar_parquet_desde_azure
from src.feature_engineering import filtrar_consistencia_localidades, preparar_dataset_enriquecido
from src.clustering import predecir_arquetipos_demanda, DISTRIBUCION_ESPERADA_ARQUETIPOS, cargar_modelo_clustering
from src.jerarquia import predecir_microclusters

print(" Módulos y artefactos del repositorio cargados correctamente.")

In [ ]:
# 2. Carga Oficial desde Azure Blob Storage (Capa Gold)
blob_test_path = 'GOLD/SECUTIX/Training Data/Clustering de Localidades test/'
container_name = os.getenv('AZURE_CONTAINER_NAME', 'tuboleta')
storage_account = os.getenv('AZURE_STORAGE_ACCOUNT_NAME', 'aybdwhstorage01')

print(f"Cargando dataset de test desde: wasbs://{container_name}@{storage_account}.blob.core.windows.net/{blob_test_path}")
df_test_raw = cargar_parquet_desde_azure(container_name=container_name, blob_name=blob_test_path)
print(f"Dataset cargado: {len(df_test_raw):,} filas brutas, {len(df_test_raw.columns)} columnas.")

In [ ]:
# 3. Validación de Entrada (Los 4 Checks Obligatorios de Marcha Blanca)
print('=== [CHECK A] DUPLICADOS EXACTOS EN CRUDO ===')
n_dups_crudo = int(df_test_raw.duplicated().sum())
pct_dups_crudo = n_dups_crudo / len(df_test_raw) * 100
print(f' • Duplicados idénticos en todas las columnas: {n_dups_crudo:,} filas ({pct_dups_crudo:.2f}%)')

print('\n=== [CHECK B] CONTEO POST-FILTRO POR REGLA ===')
df_test_clean, rep_reglas = filtrar_consistencia_localidades(df_test_raw, retornar_reporte=True, verbose=True)

print('\n=== [CHECK C] EVENTOS INCOHERENTES EN QUOTA ===')
if 't_performance_id' in df_test_raw.columns and 'dn_quota' in df_test_raw.columns and 'performance_quota' in df_test_raw.columns:
    suma_q = df_test_raw.groupby('t_performance_id')['dn_quota'].transform('sum')
    incoherentes = int((suma_q != df_test_raw['performance_quota']).sum())
    print(f' • Registros con cuota de evento no coincidente en crudo: {incoherentes:,}')
else:
    incoherentes = 0
    print(' • Columnas de cuota no presentes en crudo.')

print('\n=== [CHECK D] TAMAÑO DEL DF VS. ESPERADO DE ENTRENAMIENTO ===')
n_ref = 33775
dif_pct = abs(len(df_test_clean) - n_ref) / n_ref * 100
print(f' • Registros limpios para inferencia: {len(df_test_clean):,} (Referencia entrenamiento: {n_ref:,})')
print(f' • Desviación volumétrica: {dif_pct:.2f}%')
if dif_pct > 20.0:
    print('  [ALERTA CHECK D] El tamaño difiere >20% del entrenamiento. Requiere validación de alcance de prueba.')

In [ ]:
# 4. Ruta Estándar de Enriquecimiento (preparar_dataset_enriquecido)
print('Ejecutando pipeline estándar de enriquecimiento de características...')
df_enriquecido = preparar_dataset_enriquecido(df_test_raw)
print(f'Dataset enriquecido finalizado: {len(df_enriquecido):,} localidades procesadas con éxito.')

In [ ]:
# 5. Inferencia con los Envoltorios Productivos (Cero Re-entrenamiento)
payload_v25_path = next((p for p in ['data/processed/modelo_clustering_v2_5.joblib', '../data/processed/modelo_clustering_v2_5.joblib'] if os.path.exists(p)), None)
payload_v3_path = next((p for p in ['data/processed/modelo_jerarquia_v3.joblib', '../data/processed/modelo_jerarquia_v3.joblib'] if os.path.exists(p)), None)

print(f'Cargando modelo Macro Nivel 1 v2.5: {payload_v25_path}')
payload_v25 = cargar_modelo_clustering(payload_v25_path)
pred_nivel1 = predecir_arquetipos_demanda(df_enriquecido, payload_v25)

print(f'Cargando modelo Jerárquico Nivel 2 v3.0: {payload_v3_path}')
pred_micro = predecir_microclusters(df_enriquecido, payload_v3_path)

# Integrar predicciones completas en DataFrame enriquecido
df_final_test = df_enriquecido.copy()
for c in ['arquetipo_demanda', 'micro_cluster_id', 'label_auto', 'score_confianza', 'es_frontera']:
    df_final_test[c] = pred_micro[c]

print(f'Inferencia jerárquica completada exitosamente sobre {len(df_final_test):,} filas.')
cols_muestra = [c for c in ['product', 'site', 'logical_seat_category', 'arquetipo_demanda', 'micro_cluster_id', 'label_auto', 'score_confianza', 'es_frontera'] if c in df_final_test.columns]
display(df_final_test[cols_muestra].head(15))

In [ ]:
# 6. Auditoría Cuantitativa: Distribución vs. Esperada y Calidad de Clasificación
distrib_obs = df_final_test['arquetipo_demanda'].value_counts(normalize=True).to_dict()
comparativa = []
for arq, p_esp in DISTRIBUCION_ESPERADA_ARQUETIPOS.items():
    p_obs = float(distrib_obs.get(arq, 0.0))
    comparativa.append({
        'Arquetipo': arq,
        '% Observado': round(p_obs * 100, 2),
        '% Esperado': round(p_esp * 100, 2),
        'Diferencia (pp)': round((p_obs - p_esp) * 100, 2)
    })

df_comp = pd.DataFrame(comparativa)
print('=== COMPARATIVA DE DISTRIBUCIÓN DE ARQUETIPOS (TEST DE REGRESIÓN ADAPTADO) ===')
display(df_comp)

confianza_media = float(df_final_test['score_confianza'].mean())
pct_frontera = float(df_final_test['es_frontera'].mean() * 100)
pct_vacio = float((df_final_test['logical_seat_category'].astype(str).str.strip() == '').mean() * 100)

print('\n=== MÉTRICAS CLAVE DE INCERTIDUMBRE Y CALIDAD ===')
print(f' • Score de confianza geométrico medio : {confianza_media:.4f}')
print(f' • Tasa de localidades en frontera      : {pct_frontera:.2f}% (Esperado < 20.0%)')
print(f' • Tasa de texto vacío post-filtro      : {pct_vacio:.2f}% (Esperado 0.0%)')

In [ ]:
# 7. Generación y Persistencia de Artefactos de Marcha Blanca
fecha_str = datetime.now().strftime('%Y%m%d')
rep_dir = '../reports' if os.path.exists('../reports') else 'reports'
data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
os.makedirs(rep_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

reporte_json_file = os.path.join(rep_dir, f'marcha_blanca_{fecha_str}.json')
out_csv_file = os.path.join(data_dir, 'marcha_blanca_predicciones.csv')
out_pq_file = os.path.join(data_dir, 'marcha_blanca_predicciones.parquet')

# 1. Guardar Reporte Cuantitativo JSON
reporte_dict = {
    'fecha_ejecucion': datetime.now().isoformat(),
    'path_origen': blob_test_path,
    'filas_in': len(df_test_raw),
    'filas_out': len(df_final_test),
    'checks_entrada': {
        'check_a_duplicados_crudo': n_dups_crudo,
        'check_b_conteos_por_regla': rep_reglas,
        'check_c_eventos_incoherentes_quota': incoherentes,
        'check_d_alerta_tamano_vs_entrenamiento': bool(dif_pct > 20.0),
        'check_d_diferencia_pct': round(dif_pct, 2)
    },
    'metricas_evaluacion': {
        'score_confianza_medio': round(confianza_media, 4),
        'pct_frontera': round(pct_frontera, 2),
        'pct_texto_vacio': round(pct_vacio, 2),
        'comparativa_distribucion_nivel1': {row['Arquetipo']: {'observado': row['% Observado'], 'esperado': row['% Esperado']} for row in comparativa}
    },
    'versiones_payload': {
        'nivel1_macro': 'v2.5 (modelo_clustering_v2_5.joblib)',
        'nivel2_micro': 'v3.0 (modelo_jerarquia_v3.joblib)'
    }
}

with open(reporte_json_file, 'w', encoding='utf-8') as f:
    json.dump(reporte_dict, f, indent=2, ensure_ascii=False)
print(f' [OK] Reporte JSON generado: {reporte_json_file}')

# 2. Guardar Predicciones CSV y Parquet
df_final_test.to_csv(out_csv_file, index=False)
df_final_test.to_parquet(out_pq_file, index=False)
print(f' [OK] Archivo CSV generado: {out_csv_file} ({os.path.getsize(out_csv_file)/(1024*1024):.2f} MB)')
print(f' [OK] Archivo Parquet generado: {out_pq_file} ({os.path.getsize(out_pq_file)/(1024*1024):.2f} MB)')